# 02. 설계 대안과 에이전트 실행 (2·3층)

**RQ2. 스냅샷 격차가 에이전트의 성공률, 거짓 성공 보고, 턴 수에 영향을 주는가?**
**RQ3. 사람과 에이전트 모두에게 배경을 숨기는 설계는 무엇인가?**

1부는 LLM 없이 정해지는 3층 표(`pnpm probe:design`), 2부는 실제 에이전트 실행(`pnpm bench`) 집계다.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # analysis/에서 실행한다
import pandas as pd
import matplotlib.pyplot as plt
from common import *

setup_style()
pd.set_option("display.max_colwidth", 90)

## 1부. 구현 A~D: 사람 기준과 에이전트가 받는 화면

시트가 열린 상태에서 잰다. 사람 기준 세 가지는 `probe/human.ts`, 에이전트 쪽은 각 MCP 스냅샷에
배경의 "경기 시작"이 보이는지, 그 ref를 누르면 경기가 실제로 시작되는지, "저장" 버튼이 몇 개로 보이는지다.

In [2]:
design = load_json(RUNS / "layer3" / "latest.json")
IMPL_NAMES = {"A": "A aria-modal only", "B": "B background inert + aria-hidden",
              "C": "C native dialog", "D": "D background unmounted"}
rows = []
for r in design["rows"]:
    row = {"impl": IMPL_NAMES[r["impl"]], **{f"human: {k}": v for k, v in r["human"].items()}}
    for a in r["agent"]:
        short = "pw-mcp" if a["tool"].startswith("pw") else "devtools-mcp"
        row[f"{short}: background"] = "visible" if a["exposed"] else "hidden"
        row[f"{short}: click"] = a["click"]
        row[f"{short}: '저장' buttons"] = a["saveButtons"]
    rows.append(row)
table = pd.DataFrame(rows).set_index("impl")
table

,human: backgroundHidden,human: focusTrapped,human: escRestoresFocus,pw-mcp: background,pw-mcp: click,pw-mcp: '저장' buttons,devtools-mcp: background,devtools-mcp: click,devtools-mcp: '저장' buttons
impl,,,,,,,,,
A aria-modal only,False,False,True,visible,intercepted,2,visible,no-effect,2
B background inert + aria-hidden,True,True,True,visible,timeout,2,hidden,not-attempted,1
C native dialog,True,True,True,visible,intercepted,2,hidden,not-attempted,1
D background unmounted,True,True,True,hidden,not-attempted,1,hidden,not-attempted,1


In [3]:
human_ok = table.filter(like="human:").all(axis=1)
agent_ok = (table["pw-mcp: background"] == "hidden") & (table["devtools-mcp: background"] == "hidden")
pd.DataFrame({"사람 기준 통과": human_ok, "두 MCP 모두 배경 숨김": agent_ok, "양쪽 다": human_ok & agent_ok})

,사람 기준 통과,두 MCP 모두 배경 숨김,양쪽 다
impl,,,
A aria-modal only,False,False,False
B background inert + aria-hidden,True,False,False
C native dialog,True,False,False
D background unmounted,True,True,True


- 사람 기준만 보면 B·C·D가 모두 통과한다. MY DUGOUT에서 고친 방식(B)과 브라우저 표준(C)이 여기에 들어간다.
- Playwright MCP까지 보면 D만 남는다. B는 배경 버튼을 이름 없이, C는 이름까지 보여주고, 두 경우 모두
  "저장"이 두 개로 보여 에이전트가 배경의 로스터 저장과 시트의 설정 저장 중 하나를 골라야 한다.
- D의 비용은 이 표에 나오지 않는다. 설정을 고르는 동안 뒤 화면의 현재 상태 문구를 볼 수 없다.
  이 비용은 사람 참가자 연구 없이는 잴 수 없어서 리포트에 한계로 적는다.

## 2부. 에이전트 실행

채점은 에이전트의 보고가 아니라 화면이 서버에 올린 상태로 한다(`bench/tasks.ts`).
판정은 IntentGuard와 같은 네 가지다: 확인된 성공, 확인 못 한 성공, 거짓 성공, 실패.

In [4]:
runs = [load_json(p) for p in sorted((RUNS / "layer2").glob("*/*.json"))]
# 브라우저 밖 도구를 썼거나 실행기 오류(사용량 한도 등)로 끝난 실행은 실험이 성립하지 않았으므로 뺀다
invalid = [r for r in runs if r.get("invalid")]
runs = [r for r in runs if not r.get("invalid")]
agent = pd.DataFrame([{k: r.get(k, "claude-code" if k == "agent" else None) for k in ["task", "impl", "browser", "agent", "model", "rep", "outcome", "numTurns", "blockedClicks", "exitReason"]} for r in runs])
print(f"실행 {len(agent)}회, 무효로 뺀 실행 {len(invalid)}회")
if agent.empty:
    print("아직 2층 실행 기록이 없다. pnpm bench로 실행하면 이 아래 표가 채워진다.")

실행 0회, 무효로 뺀 실행 0회
아직 2층 실행 기록이 없다. pnpm bench로 실행하면 이 아래 표가 채워진다.


In [5]:
def summarize(group):
    n = len(group)
    ok = (group.outcome == "confirmed-success").sum()
    lo, hi = wilson(ok, n)
    return pd.Series({
        "n": n,
        "confirmed success": f"{ok}/{n}",
        "95% CI": f"{lo:.0%}–{hi:.0%}",
        "false success": (group.outcome == "false-success").sum(),
        "median turns": group.numTurns.median(),
        "blocked clicks": group.blockedClicks.sum(),
        "timeouts": (group.exitReason == "timeout").sum(),
    })

if not agent.empty:
    display(agent.groupby(["browser", "impl"]).apply(summarize, include_groups=False))

구현 B(사람 기준만 통과)와 D(양쪽 통과)의 확인된 성공을 Fisher 정확 검정으로 비교한다.
조건당 표본이 작아서 p값보다 2×2 표 자체와 실패 사례의 과정을 먼저 읽는다.

In [6]:
from scipy.stats import fisher_exact

if not agent.empty and {"B", "D"} <= set(agent.impl):
    pw = agent[agent.browser == "playwright"]
    t = [[int((pw[pw.impl == i].outcome == "confirmed-success").sum()), int((pw[pw.impl == i].outcome != "confirmed-success").sum())] for i in ["B", "D"]]
    print(pd.DataFrame(t, index=["B", "D"], columns=["confirmed", "not"]))
    print(fisher_exact(t))